# BCIC2A 最终提交 Notebook

这份 notebook 是 BCIC2A 四分类脑电运动想象任务的最终提交代码整理版。最终路线：subject-aware 预处理 + ATCNet / EEGNet / TCN 家族 + 稳健融合。我们探索过 foundation model，但在本数据集上的表现不如传统模型，因此最终没有采用。


In [ ]:
from pathlib import Path
import json
import math
import random
import hashlib

import h5py
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except Exception as exc:
    torch = None
    nn = None
    F = None
    Dataset = object
    DataLoader = None
    print("PyTorch import failed; final txt writing cells still work.", repr(exc))

ROOT = Path.cwd()
DATA_NAME = "BCIC2A"
DATA_DIR = ROOT / "course_project" / DATA_NAME
INDEX_PATH_TRAIN = DATA_DIR / "train.h5"
INDEX_PATH_VAL = DATA_DIR / "val.h5"
INDEX_PATH_TEST = DATA_DIR / "test_x_only.h5"
OUTPUT_PATH = DATA_DIR / f"{DATA_NAME}.txt"

N_SUBJECTS = 9
N_CLASSES = 4
CHANS = 22
TIME_POINT = 200

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
print("Root:", ROOT)
print("Output:", OUTPUT_PATH)


## 1. 数据检查

数据以 HDF5 形式保存。本课程划分下，训练集有 720 个 trial，验证集和测试集各有 360 个 trial。代码会检查样本规模、标签分布和输入维度，确保后续训练与写出的提交文件格式正确。


In [ ]:
def describe_h5(path, has_label=True):
    with h5py.File(path, "r") as f:
        print("\n", path)
        print("keys:", list(f.keys()))
        print("X:", f["X"].shape, f["X"].dtype)
        if has_label:
            print("y:", f["y"].shape, f["y"].dtype)
            y = f["y"][()]
            print("label counts:", np.bincount(y.astype(int), minlength=N_CLASSES).tolist())

describe_h5(INDEX_PATH_TRAIN, has_label=True)
describe_h5(INDEX_PATH_VAL, has_label=True)
describe_h5(INDEX_PATH_TEST, has_label=False)


## 2. Subject-Aware 预处理

最稳定、最明确的提升来自按 subject 分别做通道标准化。不同被试之间的 EEG 分布差异很大，直接全局标准化会把 subject 差异混进分类问题里；按 subject 做 `channel z-score` 可以减轻这种分布偏移，同时保留每个 trial 内的运动想象信号。


In [ ]:
def infer_subjects(n, n_subjects=N_SUBJECTS):
    if n % n_subjects != 0:
        raise ValueError(f"样本数 n={n} 不能均匀划分为 {n_subjects} 组")
    block = n // n_subjects
    return np.repeat(np.arange(1, n_subjects + 1), block)

class SubjectChannelZ:
    def __init__(self, eps=1e-6):
        self.eps = eps
        self.stats = {}

    def fit(self, x, subjects):
        # x: numpy array, shape (N, C, T)
        for sid in sorted(np.unique(subjects)):
            xs = x[subjects == sid]
            mean = xs.mean(axis=(0, 2), keepdims=True)
            std = xs.std(axis=(0, 2), keepdims=True)
            self.stats[int(sid)] = (mean.astype(np.float32), np.maximum(std, self.eps).astype(np.float32))
        return self

    def transform(self, x, subjects):
        out = x.astype(np.float32).copy()
        for sid, (mean, std) in self.stats.items():
            mask = subjects == sid
            if mask.any():
                out[mask] = (out[mask] - mean) / std
        return out

def trial_z(x, eps=1e-6):
    mean = x.mean(axis=2, keepdims=True)
    std = np.maximum(x.std(axis=2, keepdims=True), eps)
    return (x - mean) / std

class BCIC2ADataset(Dataset):
    def __init__(self, path, has_label=True, preprocessor=None):
        with h5py.File(path, "r") as f:
            x = f["X"][()].astype(np.float32)
            y = f["y"][()].astype(np.int64) if has_label else None
        subjects = infer_subjects(len(x))
        if preprocessor is not None:
            x = preprocessor.transform(x, subjects)
        x = trial_z(x).astype(np.float32)
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)
        self.subjects = torch.tensor(subjects, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x[idx], self.subjects[idx]
        return self.x[idx], self.y[idx], self.subjects[idx]

# 在训练集上拟合 subject-aware 标准化参数。
with h5py.File(INDEX_PATH_TRAIN, "r") as f:
    train_x_raw = f["X"][()].astype(np.float32)
train_subjects = infer_subjects(len(train_x_raw))
subject_z = SubjectChannelZ().fit(train_x_raw, train_subjects)
print("Subject stats fitted:", sorted(subject_z.stats.keys()))


## 3. 主要模型家族

最终实验主要使用三类传统模型。ATCNet 上限最高，EEGNet Serious 参数很少但非常稳定，Multiscale-TCN 单模型分数较低但错误结构有互补价值。下面给出的是 EEGNet Serious 的紧凑实现；ATCNet / TCN 变体在实验脚本中训练，并将 logits 用于最终融合。


In [ ]:
class EEGNetSerious(nn.Module):
    def __init__(self, chans=CHANS, num_classes=N_CLASSES, time_point=TIME_POINT, f1=4, d=4, pk1=4, pk2=8, dp=0.7, conv_ks=64, sep_ks=16):
        super().__init__()
        f2 = f1 * d
        self.block1 = nn.Sequential(
            nn.Conv2d(1, f1, (1, conv_ks), padding=(0, conv_ks // 2), bias=False),
            nn.BatchNorm2d(f1),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(f1, d * f1, (chans, 1), groups=f1, bias=False),
            nn.BatchNorm2d(d * f1),
            nn.ELU(),
            nn.AvgPool2d((1, pk1), stride=pk1),
            nn.Dropout(dp),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(d * f1, f2, (1, sep_ks), groups=f2, bias=False, padding=(0, sep_ks // 2)),
            nn.Conv2d(f2, f2, kernel_size=1, bias=False),
            nn.BatchNorm2d(f2),
            nn.ELU(),
            nn.AvgPool2d((1, pk2), stride=pk2),
            nn.Dropout(dp),
        )
        self.classifier = nn.Linear(f2 * ((time_point // pk1) // pk2), num_classes)

    def forward(self, x):
        x = self.block1(x.unsqueeze(1))
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x.flatten(1))

def try_build_atcnet():
    # 可选：如果提交时包含 experiments 源码目录，这里可以直接构建 ATCNet。
    import sys
    exp_dir = ROOT / "experiments"
    teammate_dir = exp_dir / "teammate_repo"
    sys.path.insert(0, str(exp_dir))
    sys.path.insert(0, str(teammate_dir))
    from fast_iter_bcic2a import atc_kwargs
    from model.atcnet import ATCNet
    kwargs = atc_kwargs(kernel_tcn=6, n_windows=5, tcn_depth=2)
    return ATCNet(chans=CHANS, num_classes=N_CLASSES, time_point=TIME_POINT, **kwargs)

if torch is not None:
    eegnet = EEGNetSerious()
    print("EEGNet Serious trainable params:", sum(p.numel() for p in eegnet.parameters() if p.requires_grad))


## 4. 训练与验证工具

最终项目不是依赖单个 seed，而是训练多个配置和 seed，再根据验证集准确率、分 subject 表现和模型互补性筛选。下面这个训练循环用于复现单个 EEGNet Serious 模型。


In [ ]:
def per_subject_accuracy(pred, y, subjects):
    result = {}
    pred = np.asarray(pred)
    y = np.asarray(y)
    subjects = np.asarray(subjects)
    for sid in sorted(np.unique(subjects)):
        mask = subjects == sid
        result[int(sid)] = float((pred[mask] == y[mask]).mean())
    return result

def train_one_eegnet(epochs=300, lr=0.0018, label_smoothing=0.03, batch_size=32, seed=42):
    if torch is None:
        raise RuntimeError("PyTorch is required for training")
    seed_everything(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_ds = BCIC2ADataset(INDEX_PATH_TRAIN, True, subject_z)
    val_ds = BCIC2ADataset(INDEX_PATH_VAL, True, subject_z)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
    model = EEGNetSerious().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    best = {"acc": -1.0, "state": None, "epoch": None, "subject_acc": None}
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y, _ in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        if epoch % 10 == 0 or epoch == epochs:
            model.eval()
            preds, labels, sids = [], [], []
            with torch.no_grad():
                for x, y, subject in val_loader:
                    logits = model(x.to(device))
                    preds.extend(logits.argmax(1).cpu().numpy().tolist())
                    labels.extend(y.numpy().tolist())
                    sids.extend(subject.numpy().tolist())
            acc = float((np.asarray(preds) == np.asarray(labels)).mean())
            if acc > best["acc"]:
                best = {
                    "acc": acc,
                    "state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                    "epoch": epoch,
                    "subject_acc": per_subject_accuracy(preds, labels, sids),
                }
            if epoch % 50 == 0 or epoch == epochs:
                print(f"epoch={epoch:04d} val_acc={acc:.4f} best={best['acc']:.4f}")
    model.load_state_dict(best["state"])
    return model, best

# Example for reproduction, intentionally not run by default:
# model, best = train_one_eegnet(epochs=900, lr=0.0018, label_smoothing=0.03, seed=42)
# best


## 5. 融合原则

最终提交不是直接选择某一个单模型，而是融合后的结果。整体流程如下：

1. 训练 subject-aware ATCNet、EEGNet Serious，以及少量互补 TCN 模型。
2. 保留表现稳定的旧 soft-fusion 池，因为它在验证集上的弱 subject 表现比很多新高分单模型更稳。
3. 根据验证集上的 subject 诊断和混淆矩阵，降低弱 subject 上单一模型失效带来的风险。
4. 只在有验证集和嵌套审计支持的位置使用 subject-aware router，避免为了单个验证分数过度调参。
5. 最后保留一个高一致性模型共识修正，用于修正融合结果中最不稳定的少数样本。

为了避免重新训练带来的随机性，下面直接固化最终标签，确保重新运行 notebook 时能复现完全一致的提交 txt。


In [ ]:
def write_submission(labels, output_path=OUTPUT_PATH):
    labels = [int(x) for x in labels]
    assert len(labels) == 360, f"expected 360 labels, got {len(labels)}"
    assert set(labels).issubset({0, 1, 2, 3}), sorted(set(labels))
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for label in labels:
            f.write(f"{int(label)}\n")
    return output_path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest().upper()


## 6. 最终预测与 TXT 生成

下面的标签来自最终冻结的融合候选。为了保证课程提交可复现，这里直接固化最终标签，并用 SHA256 校验确认写出的 txt 与最终版本一致。


In [ ]:
FINAL_LABELS = [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 2, 2, 2, 2, 1, 2, 3, 2, 2, 2, 0, 3, 1, 0, 0, 0, 2, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 2, 2, 1, 0, 0, 2, 1, 0, 2, 0, 1, 1, 1, 1, 3, 3, 2, 3, 1, 0, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 1, 3, 1, 3, 0, 1, 0, 3, 1, 2, 0, 0, 3, 0, 3, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 3, 1, 3, 1, 3, 3, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, 1, 0, 2, 2, 0, 2, 0, 0, 0, 0, 0, 2, 0, 2, 1, 2, 1, 1, 2, 2, 1, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 1, 1, 2, 1, 1, 1, 1, 1, 3, 2, 0, 3, 1, 3, 2, 1, 0, 0, 2, 3, 3, 3, 1, 1, 3, 1, 1, 1, 3, 3, 3, 3, 3, 0, 3, 0, 3, 3, 0, 2, 0, 2, 2, 2, 1, 1, 0, 1, 0, 0, 0, 0, 0, 3, 2, 0, 1, 0, 2, 1, 2, 2, 0, 2, 3, 2, 3, 1, 3, 0, 0, 2, 0, 0, 0, 3, 2, 3, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 3, 2, 2, 2, 0, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3, 1, 2, 1, 3, 1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 2, 1, 0, 2, 1, 2, 2, 1, 3, 2, 3, 3, 3, 2, 3, 2, 2, 3, 3, 3, 2, 3, 3, 3, 3, 1, 2, 2, 2, 2, 0, 2, 2, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2]

output_path = write_submission(FINAL_LABELS, OUTPUT_PATH)
print(f"Saved {len(FINAL_LABELS)} labels to: {output_path}")
print("label counts:", np.bincount(np.asarray(FINAL_LABELS), minlength=N_CLASSES).tolist())
print("first 10:", FINAL_LABELS[:10])
print("last 10:", FINAL_LABELS[-10:])
generated_hash = sha256_file(output_path)
print("sha256:", generated_hash)
EXPECTED_SHA256 = "BA9655F799146D0C63B0DB471C14A5E1E63D1AF4AE8CEA22E6D1A56523EE408A"
assert generated_hash == EXPECTED_SHA256


## 7. 格式检查

这一格复刻最终提交前的格式检查：输出必须恰好有 360 个非空标签，标签只能是 `0..3`，并且文件末尾有换行。


In [ ]:
raw = OUTPUT_PATH.read_bytes()
lines = OUTPUT_PATH.read_text(encoding="utf-8").splitlines()
labels_from_file = [int(x.strip()) for x in lines if x.strip()]
print("lines:", len(lines))
print("empty lines:", sum(1 for x in lines if x.strip() == ""))
print("valid label set:", sorted(set(labels_from_file)))
print("ends with LF:", raw.endswith(b"\n"))
assert labels_from_file == FINAL_LABELS
assert len(labels_from_file) == 360
assert set(labels_from_file).issubset({0, 1, 2, 3})
assert raw.endswith(b"\n")
print("Format check passed.")
